In [3]:
!pip install transformers torch scikit-learn

import pandas as pd
import numpy as np
import torch

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import BertTokenizer, AutoModelForSequenceClassification

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

In [11]:
from google.colab import files

uploaded = files.upload()

for filename in uploaded.keys():
  print(f'User uploaded file "{filename}" with length {len(uploaded[filename])} bytes')


Saving twitter_training.csv to twitter_training.csv
User uploaded file "twitter_training.csv" with length 10325088 bytes


In [4]:
df = pd.read_csv("twitter_training.csv", header=None)

df.columns = ['id', 'topic', 'sentiment', 'text']

df = df[['text', 'sentiment']]
df = df.dropna()

df.head()

,text,sentiment
0,im getting on borderlands and i will murder yo...,Positive
1,I am coming to the borders and I will kill you...,Positive
2,im getting on borderlands and i will kill you ...,Positive
3,im coming on borderlands and i will murder you...,Positive
4,im getting on borderlands 2 and i will murder ...,Positive


In [5]:
# Convert labels
df['sentiment'] = df['sentiment'].map({
    'Negative': 0,
    'Neutral': 1,
    'Positive': 2
})

df = df.dropna()
df['sentiment'] = df['sentiment'].astype(int)

# Lowercase
df['text'] = df['text'].str.lower()

df = df.sample(800)

texts = df['text'].tolist()
labels = df['sentiment'].tolist()

In [6]:
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    texts, labels, test_size=0.3, random_state=42
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, random_state=42
)

In [7]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

train_encodings = tokenizer(train_texts, truncation=True, padding='max_length', max_length=128)
val_encodings = tokenizer(val_texts, truncation=True, padding='max_length', max_length=128)
test_encodings = tokenizer(test_texts, truncation=True, padding='max_length', max_length=128)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [8]:
class TwitterDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [9]:
train_dataset = TwitterDataset(train_encodings, train_labels)
val_dataset = TwitterDataset(val_encodings, val_labels)
test_dataset = TwitterDataset(test_encodings, test_labels)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)
test_loader = DataLoader(test_dataset, batch_size=8)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=3
)

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)

In [11]:
optimizer = AdamW(model.parameters(), lr=2e-5)

In [12]:
epochs = 1

for epoch in range(epochs):
    print(f"\nStarting Epoch {epoch+1}")

    model.train()
    total_loss = 0

    for i, batch in enumerate(train_loader):
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if i % 20 == 0:
            print(f"Step {i}, Loss: {loss.item()}")

    print(f"Epoch {epoch+1} Loss: {total_loss}")


Starting Epoch 1
Step 0, Loss: 1.1754943132400513
Step 20, Loss: 1.0520946979522705
Step 40, Loss: 1.1527390480041504
Step 60, Loss: 0.8044304847717285
Epoch 1 Loss: 73.99111199378967


In [13]:
def evaluate(model, loader):
    model.eval()
    predictions, true_labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits

            preds = torch.argmax(logits, dim=1)

            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

    print("Classification Report:\n")
    print(classification_report(true_labels, predictions))

    print("\nConfusion Matrix:\n")
    print(confusion_matrix(true_labels, predictions))

In [14]:
evaluate(model, test_loader)

Classification Report:

              precision    recall  f1-score   support

           0       0.60      0.76      0.67        46
           1       0.54      0.19      0.29        36
           2       0.59      0.76      0.67        38

    accuracy                           0.59       120
   macro avg       0.58      0.57      0.54       120
weighted avg       0.58      0.59      0.55       120


Confusion Matrix:

[[35  2  9]
 [18  7 11]
 [ 5  4 29]]


In [15]:
for param in model.bert.parameters():
    param.requires_grad = False

In [16]:
df = df.sample(800)
max_length = 128
batch_size = 8
epochs = 1